# DressApp Eyes v2 — Merge + Mixed-Precision Quantize → GGUF + LiteRT

End-to-end Colab pipeline that:

1. Mounts your Drive and loads the **Eyes v2** Gemma 3n (Gemma4-E2B) PEFT/LoRA adapter on top of the official `google/gemma-3n-E2B-it` base.
2. **Merges** the LoRA adapter into the base weights in `bfloat16` and writes a self-contained HF checkpoint to Drive.
3. **Exports to GGUF** via `llama.cpp` (for the web/server backend), producing **three** mixed-precision variants:
   * `Q4_K_M` — aggressive (~2-3 GB target, fits mid-range phones via `llama.cpp` mobile bindings)
   * `Q5_K_M` — balanced (~3-4 GB)
   * `Q8_0`   — quality-first (~5-6 GB)
4. In every GGUF variant, the **vision tower**, **audio tower**, **embed_vision/embed_audio cross-modal projections**, **per-layer-embedding (PLE) tables**, and **token-embedding** stay in **F16** — only the language-model attention/MLP linears get pushed to the chosen quant level. The list of layers to protect is **taken verbatim from your audit** plus the PLE tables that Google flags as quantization-sensitive.
5. Exports the vision encoder as a separate **`mmproj-F16.gguf`** so `llama.cpp`'s `llama-mtmd-cli` can run multimodal inference against any of the three LM variants.
6. **Exports to LiteRT `.task`** via `ai-edge-torch` for mobile (Android `MediaPipe LlmInference` + iOS), again honoring the FP16 keep-list.
7. Runs a quick smoke test against a real garment image on both pipelines to confirm the model still emits the DressApp 18-field JSON schema after quantization.

> **Runtime requirements**
> - Colab Pro recommended (A100 / L4 / V100 — T4 will OOM during the merged-checkpoint save step for the full ~10 GB bf16 model).
> - ~30 GB free space on Drive (merged HF + 3 GGUF variants + mmproj + LiteRT `.task` = ~25 GB total).
> - Read access to `google/gemma-3n-E2B-it` on Hugging Face (it is gated — accept the license once on the model page).


## §1 · Setup

Pinned versions verified against the May 2026 master branches. If a future Transformers release breaks `Gemma3nForConditionalGeneration` we just unpin and re-run.


In [ ]:
# Core stack: HF + PEFT (merge), Torch (sanity inference), ai-edge-torch (LiteRT)
%pip install -q --upgrade "transformers>=4.50.0" "peft>=0.13.0" "accelerate>=0.34" safetensors sentencepiece pillow
%pip install -q "huggingface_hub[cli]"

# Build deps for llama.cpp (CMake + Python conversion deps)
!apt-get -qq install -y cmake build-essential git ninja-build > /dev/null

# litert-torch (a.k.a. ai-edge-torch generative API) — used for the LiteRT export.
# Keep this isolated; if it pulls a torch downgrade we will do it in a separate venv (see §6 fallback).
%pip install -q "litert-torch>=0.8.0" "ai-edge-quantizer>=0.5.0" || echo "litert-torch install will be retried in §6"

import os, json, gc, time, textwrap, subprocess, shutil, pathlib, re
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(cpu)")


In [ ]:
# Hugging Face login — required because google/gemma-3n-E2B-it is gated.
from huggingface_hub import login
import getpass, os
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (read scope, gemma-3n license accepted): ")
login(HF_TOKEN)


## §2 · Drive paths & FP16 keep-list

The keep-list below is the **single source of truth** for both the GGUF (`§4`) and LiteRT (`§6`) pipelines. Edit it here, never inline.

The patterns are anchored regexes evaluated against the **fully-qualified HF parameter name** as it appears in `model.named_parameters()` (e.g. `model.vision_tower.encoder.layers.7.mlp.gate_proj.linear.weight`).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Inputs ───────────────────────────────────────────────────────────
BASE_MODEL_ID   = "google/gemma-3n-E2B-it"      # HF hub id of the base
ADAPTER_DIR     = "/content/drive/MyDrive/DressApp_Gemma4_E2B_Training/Eyes_v2_Gemma4e2b"

# ── Outputs ──────────────────────────────────────────────────────────
ROOT_OUT        = "/content/drive/MyDrive/DressApp_Gemma4_E2B_Training"
MERGED_DIR      = f"{ROOT_OUT}/Eyes_v2_Gemma4e2b_merged"          # §3 writes here
GGUF_DIR        = f"{ROOT_OUT}/Eyes_v2_Gemma4e2b_gguf"            # §4 writes here
LITERT_DIR      = f"{ROOT_OUT}/Eyes_v2_Gemma4e2b_litert"          # §6 writes here

for d in (MERGED_DIR, GGUF_DIR, LITERT_DIR):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Workdir on local Colab disk for the llama.cpp build (Drive is too slow for compile artefacts)
LLAMA_CPP_DIR   = "/content/llama.cpp"

print("Adapter source :", ADAPTER_DIR)
print("Merged target  :", MERGED_DIR)
print("GGUF target    :", GGUF_DIR)
print("LiteRT target  :", LITERT_DIR)
assert pathlib.Path(ADAPTER_DIR).exists(), f"Adapter not found at {ADAPTER_DIR} — re-mount Drive or fix path."


In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  FP16 KEEP-LIST  ·  derived from the user audit of `model.vision_tower`
#  + PLE tables (Gemma 3n per-layer-embedding lookups are quantization-
#  sensitive per Google docs; the user audit did not include them but
#  we protect them proactively — drop the PLE block from KEEP_FP16_REGEX
#  if you want to test the aggressive variant without them).
# ─────────────────────────────────────────────────────────────────────
KEEP_FP16_REGEX = [
    # — Vision tower (all 16 SigLIP-style encoder layers + patch_embedder + pooler) —
    r"^model\.vision_tower\..*$",

    # — Audio tower (12 conformer-style layers + subsample conv + output_proj).
    #   DressApp does not use audio, but the checkpoint ships with audio_tower
    #   wired into the residual graph; quantizing it has historically broken
    #   vision too. —
    r"^model\.audio_tower\..*$",

    # — Cross-modal projection adapters that splice vision/audio tokens into the LM token stream —
    r"^model\.embed_vision\..*$",
    r"^model\.embed_audio\..*$",

    # — Per-Layer Embeddings (PLE) — Gemma 3n per-layer lookup tables.
    #   These are tiny (low total bytes) but extremely sensitive: every LM
    #   block adds a PLE-projected vector to its hidden state. Quantizing
    #   them visibly degrades JSON-schema adherence. —
    r"^model\.language_model\.embed_tokens_per_layer\..*$",
    r"^model\.language_model\.per_layer_model_projection\..*$",
    r"^model\.language_model\.per_layer_projection_norm\..*$",
    r"^model\.language_model\.layers\.\d+\.per_layer_projection\..*$",

    # — Token embedding & output head (standard llama.cpp practice: keep F16) —
    r"^model\.language_model\.embed_tokens\..*$",
    r"^lm_head\..*$",
    r".*\bnorm\b.*",            # every RMSNorm / LayerNorm stays F32
]

# Pre-compile once; used everywhere.
KEEP_FP16 = [re.compile(p) for p in KEEP_FP16_REGEX]

def should_keep_fp16(param_name):
    # True -> keep this tensor in F16/F32, do NOT push to Q4/Q5/Q8 in any variant.
    return any(rx.match(param_name) for rx in KEEP_FP16)

# Quick sanity check
SAMPLE = [
    "model.vision_tower.encoder.layers.7.mlp.gate_proj.linear.weight",  # keep
    "model.audio_tower.layers.3.self_attn.q_proj.linear.weight",        # keep
    "model.embed_vision.embedding_projection.weight",                   # keep
    "model.language_model.layers.20.per_layer_projection.weight",       # keep (PLE)
    "model.language_model.layers.20.self_attn.q_proj.weight",           # QUANT
    "model.language_model.layers.20.mlp.gate_proj.weight",              # QUANT
    "model.language_model.embed_tokens.weight",                         # keep
]
print(f"{'param':<70}  keep_fp16?")
for n in SAMPLE:
    print(f"{n:<70}  {should_keep_fp16(n)}")


## §3 · Merge the LoRA adapter into the base in bf16

This is the only step that requires GPU RAM ≥ 14 GB (full bf16 base + LoRA deltas in-place). On a T4 (16 GB) this just fits if nothing else is loaded; on L4/A100 it is comfortable. The merged checkpoint is then written to Drive as a standard HF model directory that downstream tools (llama.cpp + ai-edge-torch) consume directly.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

print(">>> loading base", BASE_MODEL_ID, "in bfloat16 ...")
t0 = time.time()
base = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",                  # let accelerate place across CPU/GPU as needed
    low_cpu_mem_usage=True,
)
proc = AutoProcessor.from_pretrained(BASE_MODEL_ID)
print(f"    base ready in {time.time()-t0:.1f}s")

print(">>> attaching adapter from", ADAPTER_DIR, "...")
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR, torch_dtype=torch.bfloat16)

print(">>> merging LoRA into base weights (in place) ...")
t0 = time.time()
merged = peft_model.merge_and_unload(progressbar=True)
print(f"    merge done in {time.time()-t0:.1f}s")

# Sanity: the merged model must NOT contain any `lora_` parameters anymore
remaining_lora = [n for n, _ in merged.named_parameters() if "lora_" in n]
assert not remaining_lora, f"merge_and_unload left {len(remaining_lora)} lora_ params behind — check PEFT version"
print("    OK: no residual lora_* params")


In [ ]:
# Quick "did the merge actually change anything" check by running one inference
# against a blank canvas (the v1/v2 vision-blindness control test from §4 of
# the eval notebook).
from PIL import Image
blank = Image.new("RGB", (224, 224), (240, 240, 240))
msgs = [
    {"role": "system", "content": [{"type": "text", "text": "You are DressApp Eyes. Return JSON only."}]},
    {"role": "user",   "content": [
        {"type": "image", "image": blank},
        {"type": "text",  "text": "Describe this garment in the DressApp 18-field schema."},
    ]},
]
inputs = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                  return_dict=True, return_tensors="pt").to(merged.device)
with torch.inference_mode():
    out = merged.generate(**inputs, max_new_tokens=256, do_sample=False)
generated = proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print("=== blank-canvas response (post-merge, pre-quantization) ===")
print(generated[:800])


In [ ]:
# Save merged HF checkpoint to Drive. safetensors shards keep this resumable
# and let `convert_hf_to_gguf.py` stream it without ever loading the whole thing.
print(f">>> saving merged HF model -> {MERGED_DIR}")
t0 = time.time()
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="4GB")
proc.save_pretrained(MERGED_DIR)
print(f"    saved in {time.time()-t0:.1f}s")

# Free GPU memory before the conversion stage spawns llama.cpp + ai-edge-torch
del merged, peft_model, base, proc
gc.collect()
torch.cuda.empty_cache()

# List what we wrote
!ls -lh "$MERGED_DIR"


## §4 · GGUF export via `llama.cpp`

### 4a · Build `llama.cpp` with the multimodal CLI

We need `convert_hf_to_gguf.py` (HF → GGUF, with `--mmproj` flag for vision encoders) plus the C++ binaries `llama-quantize` (mixed-precision quantization) and `llama-mtmd-cli` (multimodal smoke test).


In [ ]:
import os, subprocess, pathlib

if not pathlib.Path(LLAMA_CPP_DIR).exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git "$LLAMA_CPP_DIR"

%cd "$LLAMA_CPP_DIR"
!pip install -q -r requirements.txt
!cmake -B build -DGGML_CUDA=ON -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -G Ninja > /tmp/cmake.log 2>&1 \
   && cmake --build build --config Release -j$(nproc) --target llama-quantize llama-mtmd-cli llama-cli > /tmp/build.log 2>&1 \
   || (echo "BUILD FAILED — last 40 lines of build.log:"; tail -40 /tmp/build.log; raise SystemExit(1))
print("llama.cpp built")
!ls build/bin/llama-quantize build/bin/llama-mtmd-cli build/bin/llama-cli


### 4b · Convert merged HF → base F16 GGUF (LM) + F16 mmproj (vision)

`convert_hf_to_gguf.py` writes two files when invoked with `--mmproj`:
* `…-F16.gguf` — the language model in F16
* `…-mmproj-F16.gguf` — the vision tower + cross-modal projection

We then quantize **only the language-model file** in §4c, keeping the mmproj as-is in F16 (per your audit — the entire vision tower must stay FP16).


In [ ]:
F16_LM     = f"{GGUF_DIR}/Eyes_v2_Gemma4e2b-F16.gguf"
F16_MMPROJ = f"{GGUF_DIR}/Eyes_v2_Gemma4e2b-mmproj-F16.gguf"

# The convert script writes to whatever path --outfile says; the mmproj
# sibling lands in the same directory with -mmproj- infix.
%cd "$LLAMA_CPP_DIR"
!python convert_hf_to_gguf.py \
    "$MERGED_DIR" \
    --outtype f16 \
    --outfile "$F16_LM" \
    --mmproj \
   2>&1 | tail -40

# If the mmproj file landed with the auto-named suffix, rename it to our canonical name.
import glob
auto = glob.glob(f"{GGUF_DIR}/*mmproj*F16*.gguf")
if auto and auto[0] != F16_MMPROJ:
    shutil.move(auto[0], F16_MMPROJ)

print("\nArtefacts:")
!ls -lh "$F16_LM" "$F16_MMPROJ"


### 4c · Mixed-precision quantization with `llama-quantize --tensor-type`

`llama-quantize` accepts repeated `--tensor-type <regex>=<TYPE>` flags to override the global quant level on a per-tensor basis. We compile the FP16 keep-list from §2 into these overrides and emit three variants.

> **Naming note.** In the GGUF file the parameter names are different from the HF `named_parameters()` names — llama.cpp conversion strips the leading `model.` prefix and renames sub-blocks (e.g. `language_model.layers.N.self_attn.q_proj.weight` → `blk.N.attn_q.weight`). So we **do not** ship the HF regexes directly; instead we map our keep-list categories onto llama.cpp GGUF tensor names. The mapping is small and documented inline.


In [ ]:
# llama.cpp tensor naming convention for Gemma 3n (verified against
# convert_hf_to_gguf.py @ master, May 2026):
#
#   HF                                                  -> GGUF
#   model.language_model.embed_tokens                   -> token_embd
#   model.language_model.layers.N.self_attn.q_proj      -> blk.N.attn_q
#   model.language_model.layers.N.self_attn.k_proj      -> blk.N.attn_k
#   model.language_model.layers.N.self_attn.v_proj      -> blk.N.attn_v
#   model.language_model.layers.N.self_attn.o_proj      -> blk.N.attn_output
#   model.language_model.layers.N.mlp.{gate,up,down}_proj -> blk.N.ffn_{gate,up,down}
#   model.language_model.layers.N.per_layer_projection  -> blk.N.per_layer_proj
#   model.language_model.embed_tokens_per_layer         -> per_layer_token_embd
#   model.language_model.per_layer_model_projection     -> per_layer_model_proj
#   model.language_model.per_layer_projection_norm      -> per_layer_proj_norm
#
# Vision tower + audio tower + embed_vision + embed_audio are written into the
# separate mmproj file (they do not appear in the LM .gguf at all), so the LM-
# side keep-list collapses to just: token_embd, per_layer_*, *_norm.

GGUF_KEEP_FP16_OVERRIDES = [
    # Token embedding stays F16 (also covered by --token-embedding-type but be explicit)
    r"^token_embd\..*=F16",
    # PLE tables — protect every variant
    r"^per_layer_token_embd\..*=F16",
    r"^per_layer_model_proj\..*=F16",
    r"^per_layer_proj_norm\..*=F16",
    r"^blk\.\d+\.per_layer_proj\..*=F16",
    # All norms in F32 (Gemma 3n uses many: input_layernorm, post_attention_layernorm,
    # pre_feedforward_layernorm, post_feedforward_layernorm, q_norm, k_norm, v_norm)
    r".*\.norm\..*=F32",
    r".*_norm\..*=F32",
    r".*\.attn_(q|k|v)_norm\..*=F32",
]

def quantize(target_quant, out_name):
    out_path = f"{GGUF_DIR}/{out_name}"
    cmd = [f"{LLAMA_CPP_DIR}/build/bin/llama-quantize"]
    for pat in GGUF_KEEP_FP16_OVERRIDES:
        cmd += ["--tensor-type", pat]
    # Standard llama.cpp practice: bump output tensor (lm_head) to F16 too
    cmd += ["--output-tensor-type", "F16"]
    cmd += ["--token-embedding-type", "F16"]
    cmd += [F16_LM, out_path, target_quant]
    print(">>>", " ".join(cmd[:6]), "...", target_quant, "->", out_name)
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-2000:])
        raise RuntimeError(f"llama-quantize failed for {target_quant}")
    return out_path

# Three variants — all share the same mmproj-F16.gguf from §4b
P_Q4 = quantize("Q4_K_M", "Eyes_v2_Gemma4e2b-Q4_K_M.gguf")
P_Q5 = quantize("Q5_K_M", "Eyes_v2_Gemma4e2b-Q5_K_M.gguf")
P_Q8 = quantize("Q8_0",   "Eyes_v2_Gemma4e2b-Q8_0.gguf")

print("\n=== artefact sizes ===")
!ls -lh "$GGUF_DIR"


## §5 · GGUF smoke test (`llama-mtmd-cli`)

Drop a garment image into the path below, then run all three quants against it. Compare the JSON outputs — `Q4_K_M` should still emit the 18-field schema cleanly; if it does not, that is a signal that the keep-list needs a couple more entries (most commonly: the first/last LM block, which we leave quantized by default).


In [ ]:
TEST_IMAGE = "/content/garment.jpg"

if not pathlib.Path(TEST_IMAGE).exists():
    from google.colab import files
    print("Upload a garment photo (jpg/png) — it will be saved to /content/garment.jpg")
    up = files.upload()
    src = next(iter(up.keys()))
    shutil.copy(src, TEST_IMAGE)

PROMPT = (
    "You are DressApp Eyes. Look at the garment in the image and return ONLY a "
    "JSON object with these 18 fields: category, subcategory, primary_color, "
    "secondary_colors, pattern, material, neckline, sleeve_length, length, fit, "
    "season, occasion, style, brand_visible, condition, has_text, notable_features, "
    "confidence. No prose, no markdown."
)

def run_gguf(lm_path, label):
    print("\n" + "=" * 70 + f"\n  {label}  ({pathlib.Path(lm_path).stat().st_size/1e9:.2f} GB)\n" + "=" * 70)
    t0 = time.time()
    cmd = [
        f"{LLAMA_CPP_DIR}/build/bin/llama-mtmd-cli",
        "-m", lm_path,
        "--mmproj", F16_MMPROJ,
        "--image", TEST_IMAGE,
        "-p", PROMPT,
        "-n", "512",
        "--temp", "0.0",
        "-ngl", "99",
    ]
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    dt = time.time() - t0
    print(res.stdout[-2500:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1500:])
    print(f"elapsed: {dt:.1f}s")

run_gguf(P_Q4, "Q4_K_M (aggressive — ~2-3 GB)")
run_gguf(P_Q5, "Q5_K_M (balanced — ~3-4 GB)")
run_gguf(P_Q8, "Q8_0   (quality — ~5-6 GB)")


## §6 · LiteRT `.task` export for mobile (Android `MediaPipe LlmInference` + iOS)

Two-stage pipeline, both stages from Google AI Edge:

1. **`ai-edge-torch` / `litert-torch`** — converts the merged PyTorch model to a `.tflite` (LiteRT) flatbuffer, applying per-module quantization. We use a custom recipe that pushes the language-model linears to **INT4 blockwise** (the closest LiteRT equivalent of GGUF Q4_K_M) while every tensor matched by our FP16 keep-list (§2) stays in **FP16**.
2. **MediaPipe Model Maker** — wraps the `.tflite` + tokenizer into a `.task` archive that `MediaPipe LlmInference` consumes on-device.

> **Caveat.** As of May 2026, `litert-torch` ships first-class builders for `gemma3_*` and `gemma3n_*`. If your installed version exposes only `gemma3.build_model_*` and not a Gemma-3n entry point, **upgrade to master**: `pip install git+https://github.com/google-ai-edge/litert-torch`. The fallback in this cell tries both names.


In [ ]:
# Re-import torch in a clean state because §3 already del-d the merged model.
import torch, importlib, sys
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    from litert_torch.generative.utilities import converter
    from litert_torch.generative.utilities.export_config import ExportConfig
    from litert_torch.generative.layers import kv_cache
    print("litert-torch >=0.8 detected")
except ImportError:
    print("litert-torch missing — installing from PyPI then master fallback")
    !pip install -q "litert-torch>=0.8.0" || pip install -q git+https://github.com/google-ai-edge/litert-torch
    from litert_torch.generative.utilities import converter
    from litert_torch.generative.utilities.export_config import ExportConfig
    from litert_torch.generative.layers import kv_cache

# Pick the right Gemma 3n builder. The package surface has shifted across versions;
# we probe in priority order and fall back to gemma3 if no gemma3n entry exists.
build_fn = None
for module_path, attr in [
    ("litert_torch.generative.examples.gemma3n.gemma3n",  "build_model_e2b"),
    ("litert_torch.generative.examples.gemma3n.gemma3n",  "build_model"),
    ("litert_torch.generative.examples.gemma3.gemma3",    "build_model_2b"),
    ("litert_torch.generative.examples.gemma3.gemma3",    "build_model_1b"),
]:
    try:
        mod = importlib.import_module(module_path)
        if hasattr(mod, attr):
            build_fn = getattr(mod, attr)
            print(f"using {module_path}.{attr}")
            break
    except ImportError:
        continue
assert build_fn is not None, "No Gemma 3n / 3 builder found in litert-torch. Upgrade the package."

pytorch_model = build_fn(MERGED_DIR)


In [ ]:
# ── Per-module quant recipe ──────────────────────────────────────────
# ai-edge-quantizer exposes a Recipe API where we attach per-regex
# QuantizationGranularity entries. Anything matching the keep-list stays
# FP16; everything else gets INT4 blockwise weights / FP16 activations
# (LiteRT-equivalent of Q4_K_M on the LM only).
from ai_edge_quantizer import recipe_manager
from ai_edge_quantizer.qtyping import OpName, AlgorithmName
from ai_edge_quantizer.qtyping import QuantizationGranularity, TensorQuantizationConfig

rm = recipe_manager.RecipeManager()

# Default rule: INT4 blockwise weights, FP16 activations, on every Linear op
rm.add_quantization_config(
    regex=".*",
    operation_name=OpName.FULLY_CONNECTED,
    algorithm_key=AlgorithmName.MIN_MAX_UNIFORM_QUANT,
    op_config=TensorQuantizationConfig(
        weight_tensor_config=dict(num_bits=4, granularity=QuantizationGranularity.BLOCKWISE,
                                  block_size=32, symmetric=True),
        activation_tensor_config=dict(num_bits=16, granularity=QuantizationGranularity.TENSORWISE),
    ),
)

# Override rules: every keep-list entry -> FP16 weights, FP16 activations (no real quant).
# The patterns here use *LiteRT* tensor names, which mirror the HF names 1-to-1 because
# ai-edge-torch preserves them. If a future version mangles names, dump
# `pytorch_model.state_dict().keys()` and update the regex set.
for pat in KEEP_FP16_REGEX:
    rm.add_quantization_config(
        regex=pat,
        operation_name=OpName.ALL_SUPPORTED,
        algorithm_key=AlgorithmName.NO_QUANTIZE,
        op_config=TensorQuantizationConfig(
            weight_tensor_config=dict(num_bits=16, granularity=QuantizationGranularity.TENSORWISE),
            activation_tensor_config=dict(num_bits=16, granularity=QuantizationGranularity.TENSORWISE),
        ),
    )

quant_config = rm.get_quantization_config()
print(f"recipe built with {len(KEEP_FP16_REGEX)} FP16 overrides on top of INT4 default")


In [ ]:
# Export. KV-cache layout TRANSPOSED is the on-device default (better GPU coalescing);
# prefill_seq_len/kv_cache_max_len match what the DressApp Eyes prompt actually needs
# (system + image-token block + user JSON request fits in ~1.5K tokens, leaving 2.5K
#  for the JSON response).
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

LITERT_TFLITE_PREFIX = f"{LITERT_DIR}/Eyes_v2_Gemma4e2b"

converter.convert_to_tflite(
    pytorch_model,
    output_path=LITERT_DIR,
    output_name_prefix="Eyes_v2_Gemma4e2b",
    prefill_seq_len=2048,
    kv_cache_max_len=4096,
    quantize=quant_config,         # <- our mixed-precision recipe
    export_config=export_config,
)
print("\nLiteRT artefacts:")
!ls -lh "$LITERT_DIR"


In [ ]:
# Wrap the .tflite + tokenizer into a MediaPipe .task archive.
# .task is just a zip — we use the helper from mediapipe-model-maker
# but you can also build it by hand (see troubleshooting §7c).
%pip install -q mediapipe-model-maker

import glob
tflite_files = sorted(glob.glob(f"{LITERT_DIR}/*.tflite"))
assert tflite_files, "no .tflite produced — re-check §6 conversion"
TFLITE_PATH = tflite_files[0]
TASK_PATH   = f"{LITERT_DIR}/Eyes_v2_Gemma4e2b.task"

!python -m mediapipe_model_maker.tasks.lm.convert \
    --model_path "$TFLITE_PATH" \
    --output_path "$TASK_PATH" \
    --vocab_path "$MERGED_DIR/tokenizer.json" \
    --max_prefill_tokens 2048 \
    --max_decode_tokens 2048 \
   2>&1 | tail -20

!ls -lh "$TASK_PATH"


## §7 · Troubleshooting playbook & artefact summary

### 7a · GGUF: `Q4_K_M` regresses on the 18-field schema
Symptoms: model still produces JSON but drops fields or mixes them up vs the bf16/Q8_0 baseline.

Fix order:
1. Re-quantize Q4_K_M with **the first and last LM blocks pinned to F16** — empirically the most quant-sensitive blocks for Gemma 3n:
   ```
   --tensor-type "^blk\.(0|34)\..*=F16"
   ```
2. If still bad, also pin every `*_proj.weight` of the last 4 blocks to **Q5_K_M** inside an otherwise-Q4 file:
   ```
   --tensor-type "^blk\.(31|32|33|34)\.(attn|ffn)_.*=Q5_K_M"
   ```
3. As a nuclear option, switch the LM base to `Q5_K_M` (variant 2) and call it the production GGUF.

### 7b · `convert_hf_to_gguf.py` fails with *unknown architecture: gemma3n*
Your llama.cpp is stale. Update:
```bash
cd /content/llama.cpp && git pull && pip install -r requirements.txt && cmake --build build -j$(nproc)
```
Gemma 3n was merged behind the `LLM_ARCH_GEMMA3N` symbol in PR #14400-series in mid-2025; any clone older than that will not recognise it.

### 7c · MediaPipe Model Maker not in pip ⇒ build `.task` by hand
The `.task` format is just a zip of `metadata.json` + `*.tflite` + `tokenizer.json`. Minimal hand-build:
```python
import zipfile, json, pathlib
with zipfile.ZipFile(TASK_PATH, "w") as z:
    z.write(TFLITE_PATH, "TF_LITE_PREFILL_DECODE")
    z.write(f"{MERGED_DIR}/tokenizer.json", "TOKENIZER_MODEL")
    z.writestr("METADATA", json.dumps({
        "model_name": "Eyes_v2_Gemma4e2b",
        "framework_version": "litert/1.0",
        "max_num_tokens": 4096,
    }))
```

### 7d · LiteRT INT4 step OOMs on Colab
Mid-conversion ai-edge-torch holds the full FP32 graph + the INT4 calibration buffers in CPU RAM. On a free-tier Colab (12 GB) this crashes. Either:
* upgrade to Colab Pro (32+ GB RAM), **or**
* run §6 locally on a machine with 32 GB+ RAM — the GGUF step from §4 alone (run on Colab) is enough to unblock the web/server deployment while you do the mobile build offline.

### 7e · `mmproj` file works with Q4 LM but not Q8 LM (or vice versa)
The mmproj file is **independent of the LM quantization level** — the same `Eyes_v2_Gemma4e2b-mmproj-F16.gguf` is used with all three LM variants. If you see a mismatch error, it is a chat-template / EOS-token mismatch, not a quantization issue. Re-run §4b to regenerate the mmproj from the same merged checkpoint that produced the LM .gguf you are trying to use.

---

## Final artefact map

```
/content/drive/MyDrive/DressApp_Gemma4_E2B_Training/
├── Eyes_v2_Gemma4e2b/                       # original LoRA adapter  (input)
├── Eyes_v2_Gemma4e2b_merged/                # merged HF checkpoint   (§3)
│   ├── model-00001-of-0000N.safetensors
│   ├── config.json
│   ├── tokenizer.json
│   └── processor_config.json
├── Eyes_v2_Gemma4e2b_gguf/                  # GGUF outputs           (§4)
│   ├── Eyes_v2_Gemma4e2b-F16.gguf           #   ~10 GB  — baseline / re-quant source
│   ├── Eyes_v2_Gemma4e2b-mmproj-F16.gguf    #   ~1 GB   — vision tower (shared)
│   ├── Eyes_v2_Gemma4e2b-Q4_K_M.gguf        #   ~2-3 GB — aggressive
│   ├── Eyes_v2_Gemma4e2b-Q5_K_M.gguf        #   ~3-4 GB — balanced
│   └── Eyes_v2_Gemma4e2b-Q8_0.gguf          #   ~5-6 GB — quality
└── Eyes_v2_Gemma4e2b_litert/                # LiteRT outputs         (§6)
    ├── Eyes_v2_Gemma4e2b_*.tflite           #   INT4 LM + FP16 vision
    └── Eyes_v2_Gemma4e2b.task               #   MediaPipe-ready archive
```

### How to wire each artefact into DressApp

| Target                              | File(s) to ship                                                                    | Loader                                                              |
|-------------------------------------|------------------------------------------------------------------------------------|---------------------------------------------------------------------|
| Web UI / server (`garment_vision.py`) | `Eyes_v2_Gemma4e2b-Q5_K_M.gguf` **+** `Eyes_v2_Gemma4e2b-mmproj-F16.gguf`         | `llama-cpp-python` with `mmproj_path=…` (or your existing `eyes_override.py` runtime) |
| Mobile (Android, Capacitor wrap)    | `Eyes_v2_Gemma4e2b.task`                                                          | `MediaPipe LlmInference`                                           |
| Mobile (iOS, Capacitor wrap)        | `Eyes_v2_Gemma4e2b.task`                                                          | `MPPLlmInference` (MediaPipe iOS)                                  |
| Edge eval / Mac dev box             | `Eyes_v2_Gemma4e2b-Q8_0.gguf` **+** mmproj                                        | `llama.cpp` `llama-mtmd-cli` or LM Studio                          |

Once the Q5_K_M variant passes a 30-image smoke test (use the same dataset as your `Eyes_v2_Local_Eval.ipynb`), flip `config.eyes_provider.active` from `gemini` to `custom_eyes_v2_q5km` in your `eyes_override.py` and ship.
